# ISIC2024 후속 검증 실행 가이드 17

이 notebook은 17장 후속 benchmark를 위한 실행 가이드이자 launcher helper다.
여기서 runbook은 무엇을 먼저 점검하고, 무엇을 다음에 실행하고, 결과를 어디서 확인할지를 적어둔 운영 가이드를 뜻한다.

- `HAM10000`은 현재 비교군에서 제외한다.
- image sweep는 `RTX 4090 x 2`를 기준으로 잡는다.
- 필요하면 Hugging Face 인증에 `.env`를 사용한다.
- `RETFound`는 로컬 `checkpoints/RETFound/RETFound_cfp_weights.pth` checkpoint를 사용하므로 ready full fine-tuning pool에 포함할 수 있다.
- 실제 학습과 artifact 저장은 기존 `.py` runner가 담당한다.
- 이 notebook은 `isic2024_followup_tabular_analysis_17.ipynb`, `isic2024_followup_image_analysis_17.ipynb`, `followup_py_ipynb_split_strategy_17.md`와 함께 본다.


In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import shlex
import subprocess
import sys
from pathlib import Path

ROOT = Path("/home/junkim2603a/proj/paper_ajou_dev").resolve()
os.chdir(ROOT)


def load_dotenv(path: Path = ROOT / ".env") -> list[str]:
    loaded: list[str] = []
    if not path.exists():
        return loaded
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if not key:
            continue
        os.environ.setdefault(key, value)
        loaded.append(key)
    hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    if hf_token:
        os.environ.setdefault("HF_TOKEN", hf_token)
        os.environ.setdefault("HUGGINGFACE_HUB_TOKEN", hf_token)
    return loaded


loaded_keys = load_dotenv()
print("루트 경로:", ROOT)
print("불러온 키:", loaded_keys)
print("HF 토큰 준비 여부:", bool(os.environ.get("HF_TOKEN")))
print("파이썬 실행 파일:", sys.executable)


In [ ]:
gpu_info = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"],
    check=True,
    capture_output=True,
    text=True,
)
print(gpu_info.stdout)


## 모델 풀

후속 image 비교군은 앞에서 요청한 모델 구성을 유지하되 `HAM10000`만 제외한다.
이 notebook은 더 큰 `num_workers`와 모델별 batch size를 반영한 임시 follow-up config를 만들고, 기존 image runner를 2개 GPU에 걸쳐 실행할 수 있게 정리한다.


In [ ]:
import pandas as pd

SOURCE_CONFIG_ROOT = ROOT / "src" / "image_baselines"
FOLLOWUP_CONFIG_ROOT = ROOT / "artifacts" / "eda" / "isic2024" / "followup_image_configs_17"
FOLLOWUP_CONFIG_ROOT.mkdir(parents=True, exist_ok=True)

IMAGE_MODELS = [
    "BioMedCLIP",
    "CheXzero",
    "DeiT-S",
    "DenseNet-121",
    "DINOv2 ViT-S",
    "EfficientNet-B0",
    "EyePACS",
    "MedCLIP",
    "MONET",
    "ResNet-50",
    "RETFound",
    "TorchXRayVision",
    "ViT-B_16",
]

MODEL_BATCH_SIZE = {
    "BioMedCLIP": 8,
    "CheXzero": 8,
    "DeiT-S": 16,
    "DenseNet-121": 24,
    "DINOv2 ViT-S": 16,
    "EfficientNet-B0": 24,
    "EyePACS": 8,
    "MedCLIP": 8,
    "MONET": 4,
    "ResNet-50": 24,
    "RETFound": 4,
    "TorchXRayVision": 16,
    "ViT-B_16": 8,
}
COMMON_NUM_WORKERS = 8
CHECKPOINT_OVERRIDE = {
    "CheXzero": ROOT / "checkpoints" / "CheXzero" / "best_128_5e-05_original_22000_0.855.pt",
    "RETFound": ROOT / "checkpoints" / "RETFound" / "RETFound_cfp_weights.pth",
}


def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8-sig"))


def write_followup_configs() -> dict[str, Path]:
    created: dict[str, Path] = {}
    for model_name in IMAGE_MODELS:
        source_path = SOURCE_CONFIG_ROOT / model_name / "config.json"
        config = read_json(source_path)
        config.setdefault("dataset", {})
        config["dataset"]["num_workers"] = COMMON_NUM_WORKERS
        config["dataset"]["batch_size"] = MODEL_BATCH_SIZE.get(
            model_name,
            int(config["dataset"].get("batch_size", 8)),
        )
        if model_name in CHECKPOINT_OVERRIDE:
            config["model"]["checkpoint_path"] = str(CHECKPOINT_OVERRIDE[model_name])
        out_dir = FOLLOWUP_CONFIG_ROOT / model_name
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / "config.json"
        out_path.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
        created[model_name] = out_path
    return created


FOLLOWUP_CONFIGS = write_followup_configs()
pd.DataFrame(
    [
        {
            "model": model_name,
            "config_path": str(config_path),
            "batch_size": MODEL_BATCH_SIZE[model_name],
            "num_workers": COMMON_NUM_WORKERS,
        }
        for model_name, config_path in FOLLOWUP_CONFIGS.items()
    ]
)


In [ ]:
def has_torchxrayvision() -> bool:
    return importlib.util.find_spec("torchxrayvision") is not None


def collect_preflight_rows() -> list[dict[str, str]]:
    rows: list[dict[str, str]] = []
    for model_name in IMAGE_MODELS:
        status = "ready"
        note = ""
        if model_name == "CheXzero":
            checkpoint_path = CHECKPOINT_OVERRIDE[model_name]
            if not checkpoint_path.exists():
                status = "blocked"
                note = f"로컬 checkpoint가 없습니다: {checkpoint_path}"
            else:
                note = f"로컬 checkpoint 사용: {checkpoint_path.name}"
        elif model_name == "RETFound":
            checkpoint_path = CHECKPOINT_OVERRIDE[model_name]
            if checkpoint_path.exists():
                note = f"로컬 weight 파일 사용: {checkpoint_path.name}"
            else:
                status = "blocked"
                note = f"로컬 checkpoint가 없습니다: {checkpoint_path}"
        elif model_name == "EyePACS":
            status = "ready"
            note = "efficientnet_b3 local checkpoint를 사용하며 5-class head 대신 2-class head로 fine-tuning 합니다"
        elif model_name == "TorchXRayVision":
            if not has_torchxrayvision():
                status = "blocked"
                note = "현재 conda 환경에 torchxrayvision 패키지가 설치되어 있지 않습니다"
            else:
                status = "ready"
                note = "패키지는 설치되어 있고 backend도 연결되어 있습니다"
        elif model_name == "MedCLIP":
            status = "ready"
            note = "공식 medclip package를 사용하며 첫 실행 시 pretrained weight를 로컬 cache에 저장합니다"
        elif model_name == "MONET":
            note = "transformers를 통해 Hugging Face 모델 chanwkim/monet을 불러옵니다"
        rows.append({"model": model_name, "status": status, "note": note})
    return rows


preflight_df = pd.DataFrame(collect_preflight_rows())
preflight_df


In [ ]:
READY_IMAGE_MODELS = preflight_df.loc[preflight_df["status"] == "ready", "model"].tolist()
BLOCKED_IMAGE_MODELS = preflight_df.loc[preflight_df["status"] != "ready", ["model", "note"]]
print("실행 준비가 된 image 모델:", READY_IMAGE_MODELS)
display(BLOCKED_IMAGE_MODELS)


## 실행 명령

image sweep는 병렬 스케줄러로 `src/isic2024_benchmark/run_all_models.py`를 사용하고, 실행 대상은 ready 상태 모델로 제한한다.
이 notebook은 preflight와 runbook 계층에 머물고, 실제 실행과 재현성은 기존 multi-GPU `.py` runner가 맡는다.

추천 1차 실행 순서:

1. 먼저 `.env`와 GPU preflight 셀을 실행한다.
2. 그다음 tabular benchmark를 돌려 `strict / relaxed / oracle` 기준선을 먼저 확정한다.
3. 이어서 `ResNet-50`, `DenseNet-121`, `EfficientNet-B0`, `EyePACS`, `DeiT-S`, `ViT-B_16`로 경량 image 1차 실행을 한다.
4. 그 다음 `BioMedCLIP`, `DINOv2 ViT-S`, `MedCLIP`, `MONET`, `CheXzero`, `RETFound`로 고비용 image 2차 실행을 한다.
5. 마지막으로 `isic2024_followup_tabular_analysis_17.ipynb`와 `isic2024_followup_image_analysis_17.ipynb`에서 `pauc_above_tpr80`를 비교하고 표를 내보낸다.

이 순서를 추천하는 이유:

- tabular baseline이 더 빨리 끝나므로 가장 먼저 비교 기준선을 확보할 수 있다.
- 경량 image 단계에서 pipeline 문제를 먼저 잡아두면 고비용 backbone에 시간을 쓰기 전에 리스크를 줄일 수 있다.
- 이후 고비용 image 단계에서 `RTX 4090` 두 장을 더 안정적으로 사용할 수 있다.


In [ ]:
LIGHT_IMAGE_MODELS = [
    model_name
    for model_name in ["ResNet-50", "DenseNet-121", "EfficientNet-B0", "EyePACS", "DeiT-S", "ViT-B_16"]
    if model_name in READY_IMAGE_MODELS
]
HEAVY_IMAGE_MODELS = [
    model_name
    for model_name in ["BioMedCLIP", "DINOv2 ViT-S", "MedCLIP", "MONET", "CheXzero", "RETFound"]
    if model_name in READY_IMAGE_MODELS
]


def build_image_command(models: list[str]) -> list[str]:
    return [
        sys.executable,
        "-m",
        "isic2024_benchmark.run_all_models",
        "--config-root",
        str(FOLLOWUP_CONFIG_ROOT),
        "--dataset-root",
        str(ROOT / "dataset" / "isic-2024-challenge"),
        "--output-root",
        str(ROOT / "artifacts"),
        "--experiment-name",
        "ISIC2024-Image-Benchmark",
        "--seed",
        "42",
        "--devices",
        "0",
        "1",
        "--models",
        *models,
    ]


LIGHT_IMAGE_COMMAND = build_image_command(LIGHT_IMAGE_MODELS)
HEAVY_IMAGE_COMMAND = build_image_command(HEAVY_IMAGE_MODELS)
FULL_READY_IMAGE_COMMAND = build_image_command(READY_IMAGE_MODELS)

print("경량 image 1차 실행:")
print(shlex.join(LIGHT_IMAGE_COMMAND))
print()
print("고비용 image 2차 실행:")
print(shlex.join(HEAVY_IMAGE_COMMAND))
print()
print("ready image 전체 실행:")
print(shlex.join(FULL_READY_IMAGE_COMMAND))

RUN_IMAGE = False
IMAGE_COMMAND = FULL_READY_IMAGE_COMMAND
if RUN_IMAGE:
    env = dict(os.environ)
    env["PYTHONPATH"] = str(ROOT / "src")
    subprocess.run(IMAGE_COMMAND, check=True, cwd=ROOT, env=env)
else:
    print("선택한 image sweep를 실행하려면 RUN_IMAGE = True로 바꾸세요.")


In [ ]:
TABULAR_COMMAND = [
    sys.executable,
    "-m",
    "isic2024_benchmark.run_tabular_baselines",
    "--dataset-root",
    str(ROOT / "dataset" / "isic-2024-challenge"),
    "--eda-dir",
    str(ROOT / "artifacts" / "eda" / "isic2024"),
    "--feature-set-json",
    str(ROOT / "artifacts" / "eda" / "isic2024" / "feature_sets_recommended.json"),
    "--experiment-name",
    "ISIC2024-Tabular-Benchmark",
    "--output-root",
    str(ROOT / "artifacts" / "tabular_runs"),
    "--models",
    "xgboost",
    "catboost",
    "svm",
    "logistic_regression",
    "mlp",
]

print(shlex.join(TABULAR_COMMAND))

RUN_TABULAR = False
if RUN_TABULAR:
    env = dict(os.environ)
    env["PYTHONPATH"] = str(ROOT / "src")
    subprocess.run(TABULAR_COMMAND, check=True, cwd=ROOT, env=env)
else:
    print("tabular benchmark를 실행하려면 RUN_TABULAR = True로 바꾸세요.")
